# Phase2: 検出 — 「例から学ぶAI」で板の位置を見つける

> ⏱ 所要時間: 説明を読む 20分 ＋ 実行 約1分

## このプロジェクトのゴール（おさらい）
駐車場のナンバー読み取り精算システムのミニ版を作り、**AIはシステムの一部品**であることを確かめます。
前回（Phase1）は、**機械学習なし**で板を探し・読みました。きれいな画像ではほぼ完璧でしたが、条件が悪くなると急に壊れました（Hard で検出率0.41、全体の完全一致0%）。

## このNotebookのゴール
**人間がルールを書く代わりに、AIに「例」から板の位置を学習させ**、Phase1 の基準（ベースライン）と比べます。次の3つを確かめます。
1. 学習するAIは、**検出（見つける）**でどれだけ良くなるか
2. それでも**読み取りが失敗する**のはなぜか → **位置の精度（四隅の誤差）**が鍵
3. **学習データの作り方**（難易度の混ぜ方）で、AIはどう変わるか

## 前のNotebookとのつながり
Phase1 で、「検出の枠が少しずれると、読み取りが失敗する（IoU 0.9以上が必要）」ことが分かりました。ここでは、**四隅の位置を、ピクセル単位の精度で出せるAI**を作ります。

## キーワード
| 用語 | 意味 |
|---|---|
| **回帰** | 数値（ここでは四隅の座標）を予測すること。「分類」はグループを当てる |
| **正規化した座標** | 座標を画像の幅・高さで割って0〜1にした値。画像サイズに依存しない |
| **loss（損失）** | 予測と正解のずれの大きさ。学習では、これを小さくする方向にAIを調整する |
| **エポック** | 学習データを1周すること。ここでは「一定枚数を学習したら1エポック」と数える |
| **その場で生成（オンザフライ）** | 学習のたびに、新しい画像を作って学習すること。合成データの利点 |
| **転移学習** | 他の大量データで学習済みのAIを借りて、自分の課題に合わせて学習し直すこと |
| **四隅誤差（px）** | 予測した四隅と、正解の四隅の距離の平均（ピクセル）。小さいほど正確 |
| **Det@0.9** | IoUが0.9以上の割合。読み取りに必要な精度の目安 |
| **分布シフト** | 学習したデータと、実際に使うデータの条件が違うこと |

In [ ]:
import sys
from pathlib import Path

# 正しい環境（プロジェクトの .venv）で動いているか確認する
try:
    import torch, PIL, numpy, pandas  # noqa: F401
except ModuleNotFoundError:
    raise SystemExit(
        f"必要なライブラリが見つかりません。今のPython: {sys.executable}\n"
        "→ カーネルを「Python (plate_reader_lab)」（または プロジェクトの .venv/bin/python）に切り替えてください。"
    )

# src/ をimportできるようにする（notebooks/ からでもプロジェクト直下からでも動く）
sys.path.insert(0, str(next(p for p in [Path.cwd() / "src", Path.cwd().parent / "src"] if p.exists())))

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader

from plate_synth import generate_sample, split_seed
from plate_synth.difficulty import CLEAN
from plate_synth.scene import make_background
from plate_synth.viz import show_grid
from ai_detect.data import FileDataset, SynthDataset, corners_to_target, target_to_corners, to_tensor
from ai_detect.evaluate import corners_bbox, evaluate, mean_iou_only, predict, summarize
from ai_detect.model import make_model
from classical import read_text, rectify
from plate_eval import exact_match, iou

plt.rcParams["font.family"] = ["Hiragino Sans", "AppleGothic", "DejaVu Sans"]
%matplotlib inline
ROOT = next(p for p in [Path.cwd(), Path.cwd().parent] if (p / "src").exists())
device = "mps" if torch.backends.mps.is_available() else "cpu"
print("使用デバイス:", device)

上のセルは、道具の読み込みです。エラーが出たら、右上のカーネルを **Python (plate_reader_lab)** に切り替えてください。
「使用デバイス: mps」は、MacのGPUで計算するという意味です（cpuでも動きます）。

## 1. 課題の設定 — 画像から「8個の数」を出す

AIの仕事は次のとおりです。

- **入力**: シーン画像（320×240ピクセル、RGB）
- **出力**: 板の**四隅**（左上・右上・右下・左下）の x, y を、**0〜1に正規化した8個の数**

四隅を出せば、**傾きや斜めの歪みもまとめて**分かるので、Phase1 の OCR にそのまま渡せます（外接する四角形だけだと、傾いた板を直せません）。

> ⚠ このAIは「**必ず1枚、板がある**」と仮定しています。板がない画像でも、何かを出してしまいます（最後に確認します）。

In [ ]:
s = generate_sample(split_seed("val", "medium", 3), "medium")
t = corners_to_target(s.label["corners"])                       # 正解の四隅を、0〜1の8個の数にする
print("正解の8個の数（左上x,y / 右上x,y / 右下x,y / 左下x,y）:")
print(np.round(t.numpy(), 3))

fig, ax = plt.subplots(figsize=(5, 3.8)); ax.imshow(s.image); ax.axis("off")
c = np.asarray(s.label["corners"])
ax.plot(*np.vstack([c, c[:1]]).T, "-", color="lime", lw=2)
for name, (x, y) in zip(["左上", "右上", "右下", "左下"], c):
    ax.text(x, y, name, color="yellow", fontsize=8, ha="center", va="bottom")
ax.set_title("正解の四隅");

## 2. AIの中身 — 「位置を保ったまま」出力する

使うモデルは2種類です。

| モデル | 説明 |
|---|---|
| **自作の小さなCNN** | 畳み込みブロックを5回重ねて画像を縮め、8×10の特徴マップにする。**ゼロから**学習 |
| **ResNet18（転移学習）** | ImageNet（約100万枚）で**学習済み**の有名なCNN（前に画像分類で使ったもの）を借りて、四隅用に学習し直す |

### 大事な工夫: 平均化しない
画像分類のCNNは、最後に**画像全体を平均**して「何が写っているか」だけを答えます。でも今回は「**どこにあるか**」が必要なので、平均すると**位置の情報が消えて**しまいます。
そこで、最後の特徴マップ（8×10）を**平らに並べ直して（flatten）全結合層に渡し**、位置を保ったまま8個の数を出します。

In [ ]:
model = make_model("small")
print("パラメータ数:", f"{sum(p.numel() for p in model.parameters()):,}")
x = torch.rand(1, 3, 240, 320)
with torch.no_grad():
    h = model.norm(x)
    print("入力:", tuple(h.shape))
    for i, blk in enumerate(model.features, 1):
        h = blk(h)
        print(f"ブロック{i}の出力:", tuple(h.shape), "  ← 縦横が半分ずつになる")
    print("最後に平らに並べて全結合 →", tuple(model.head(h).shape), "（8個の数）")

**見るポイント**: ブロックを通るたびに縦横が半分になり、チャンネル数（特徴の種類）が増えます。最後は 256チャンネル × 8 × 10 = 20,480個の数になり、そこから8個を出します。

## 3. 学習データは「その場で生成」する

Phase0 で作った合成データ生成器を使い、**エポックごとに新しい画像を作って**学習します。
前に学んだ画像分類（hymenoptera）は、**243枚を何周も**見たので、丸暗記（過学習）が問題でした。今回は、**同じ画像を二度見ない**ので、丸暗記しようがありません。これが**合成データの大きな利点**です。

次のセルは、学習データの先頭8枚（正解の四隅つき）です。難易度（Easy・Medium・Hard）が順番に混ざっています。

In [ ]:
ds = SynthDataset(8, offset=0)
imgs = []
fig, axes = plt.subplots(2, 4, figsize=(13, 5.6))
for ax, k in zip(axes.flat, range(8)):
    x, y = ds[k]
    ax.imshow(x.permute(1, 2, 0).numpy()); ax.axis("off")
    c = target_to_corners(y)
    ax.plot(*np.vstack([c, c[:1]]).T, "-", color="lime", lw=1.5)
    ax.set_title(["easy", "medium", "hard"][k % 3], fontsize=9)
fig.tight_layout()
# offset を変えると、別の画像になる（同じ画像を繰り返さない仕組み）
print("offset=0 の0番目と、offset=1000 の0番目は同じ画像か:", bool((SynthDataset(8, 0)[0][0] == SynthDataset(8, 1000)[0][0]).all()))

## 4. 学習してみる（短時間版）

小さなCNNを、**短時間**（約20秒）で学習します。本実験の設定は、**20エポック × 4,500枚 = 90,000枚、3回**です（あとの表）。

### 学習のしくみ（1ステップ）
1. 画像を64枚まとめてAIに入れ、四隅（8個の数）を予測させる
2. **正解との差（loss）**を計算する（SmoothL1: 小さな差は2乗、大きな差は絶対値で扱う）
3. loss が小さくなる方向に、AIの内部の数値（パラメータ）を**少しだけ**更新する（Adam）
4. これを何千回も繰り返す

1エポックごとに、**検証用の画像**（val の先頭100枚/難易度。学習には使わない）で、平均IoUを測ります。

In [ ]:
EPOCHS, N_PER_EPOCH, BATCH, LR = 6, 1500, 64, 1e-3        # ← 演習で変えてみる
TRAIN_DIFFS = ("easy", "medium", "hard")                   # ← 演習で変えてみる

torch.manual_seed(0)
model = make_model("small").to(device)
opt = torch.optim.Adam(model.parameters(), lr=LR)
steps = EPOCHS * (N_PER_EPOCH // BATCH)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=steps, pct_start=0.15)
loss_fn = nn.SmoothL1Loss(beta=0.01)
sel = FileDataset("val", 0, 30)                             # エポックごとの確認用（学習には使わない）

ds = SynthDataset(EPOCHS * (N_PER_EPOCH // BATCH) * BATCH, offset=0, diffs=TRAIN_DIFFS)
dl = DataLoader(ds, batch_size=BATCH, shuffle=False, num_workers=4, drop_last=True)   # 画像をその場で作る
hist, tot, n, t0 = [], 0.0, 0, time.time()
model.train()
for step, (x, y) in enumerate(dl, 1):
    x, y = x.to(device), y.to(device)
    loss = loss_fn(model(x), y)
    opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    tot += loss.item() * len(y); n += len(y)
    if step % (N_PER_EPOCH // BATCH) == 0:
        v = mean_iou_only(model, sel, device); model.train()
        hist.append((step // (N_PER_EPOCH // BATCH), tot / n, v)); tot, n = 0.0, 0
        print(f"エポック{hist[-1][0]}: loss {hist[-1][1]:.4f}  val平均IoU {v:.3f}  ({time.time() - t0:.0f}秒)")

h = pd.DataFrame(hist, columns=["epoch", "loss", "val_iou"])
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(h.epoch, h.loss, "o-"); ax[0].set(title="学習 loss（小さいほど良い）", xlabel="エポック", yscale="log")
ax[1].plot(h.epoch, h.val_iou, "o-", color="C1"); ax[1].set(title="val 平均IoU（大きいほど良い）", xlabel="エポック", ylim=(0, 1))
for a in ax: a.grid(alpha=0.3)
fig.tight_layout();

**見るポイント**
- **loss が下がり**、**val のIoUが上がって**いますか？ AIが「板の位置」を学べています。
- 学習時間は**エポックの数×1500枚**に比例します。もっと長く学習すると、さらに良くなります（本実験は90,000枚）。
- **画像分類のときと違い、train と val の乖離（過学習）は目立たない**はずです（毎回新しい画像のため）。

### 学習したAIの予測を見る（評価用画像）
学習済みのAI（今学習した短時間版）で、**val の画像**（学習に使っていない101枚目以降）の四隅を予測させ、正解と重ねます（**緑=正解、赤=予測**）。

In [ ]:
ev = FileDataset("val", 100, 104)                            # 各難易度の101〜104枚目
pred = predict(model, ev, device)
fig, axes = plt.subplots(3, 4, figsize=(13, 8.4))
for ax, p, (d, path, lab) in zip(axes.flat, pred, ev.items):
    ax.imshow(Image.open(path)); ax.axis("off")
    g, q = np.asarray(lab["corners"]), target_to_corners(p)
    ax.plot(*np.vstack([g, g[:1]]).T, "-", color="lime", lw=1.5)
    ax.plot(*np.vstack([q, q[:1]]).T, "-", color="red", lw=1.5)
    ax.set_title(f"{d}  IoU={iou(corners_bbox(q), lab['bbox']):.2f}", fontsize=9)
fig.tight_layout();

## 5. 本実験の結果 — 古典的手法と比べる

短時間版では本来の力が出ないので、**本実験の結果**（20エポック×4,500枚、3回の平均±標準偏差。評価は val の101枚目以降の各400枚）を見ます。`src/run_phase2.sh` で再現できます。
「読み取り完全一致」は、**予測した四隅で板をまっすぐにして、Phase1 の OCR で読んだ**結果です（検出→読み取りの全体）。

| 難易度 | 手法 | シード数 | 検出率(IoU≥0.5) | Det@0.9 | 平均IoU | 四隅誤差(px) | 読み取り完全一致 |
|---|---|---|---|---|---|---|---|
| easy | 古典的（Phase1） | — | 0.900 | 0.772 | 0.850 | — | 0.730 |
| easy | 自作の小さなCNN（ゼロから） | 3 | 1.000 ±0.000 | 0.954 ±0.045 | 0.944 ±0.008 | 2.782 ±0.614 | 0.732 ±0.142 |
| easy | ResNet18（ImageNet学習済み） | 3 | 1.000 ±0.000 | 1.000 ±0.000 | 0.966 ±0.001 | 1.565 ±0.054 | 0.938 ±0.005 |
| easy | ResNet18（学習済み重みなし） | 1 | 1.000 | 1.000 | 0.953 | 2.274 | 0.840 |
| medium | 古典的（Phase1） | — | 0.738 | 0.395 | 0.665 | — | 0.122 |
| medium | 自作の小さなCNN（ゼロから） | 3 | 1.000 ±0.000 | 0.388 ±0.117 | 0.882 ±0.014 | 4.907 ±0.669 | 0.113 ±0.045 |
| medium | ResNet18（ImageNet学習済み） | 3 | 1.000 ±0.000 | 0.814 ±0.013 | 0.927 ±0.000 | 2.960 ±0.058 | 0.337 ±0.020 |
| medium | ResNet18（学習済み重みなし） | 1 | 1.000 | 0.572 | 0.905 | 3.746 | 0.200 |
| hard | 古典的（Phase1） | — | 0.412 | 0.043 | 0.363 | — | 0.000 |
| hard | 自作の小さなCNN（ゼロから） | 3 | 0.989 ±0.008 | 0.040 ±0.016 | 0.763 ±0.016 | 8.320 ±0.776 | 0.000 ±0.000 |
| hard | ResNet18（ImageNet学習済み） | 3 | 0.999 ±0.001 | 0.216 ±0.029 | 0.850 ±0.005 | 5.187 ±0.113 | 0.001 ±0.001 |
| hard | ResNet18（学習済み重みなし） | 1 | 0.990 | 0.115 | 0.799 | 7.002 | 0.002 |

### 何が分かるか
1. **検出（見つける）は、学習するAIが圧倒的に良い。** 検出率（IoU≥0.5）は、**全難易度でほぼ100%**（Hardでも0.99〜1.00）。古典的手法は Hard で 0.41 でした。しかも、**しきい値の手調整なし**で、1つのモデルが全条件に対応しています。
2. **ところが、読み取りは別の話。** 検出率が100%でも、**読み取り完全一致は Easy でも 0.73〜0.94、Medium で 0.11〜0.34、Hard はほぼ 0** です。
   - **鍵は四隅の誤差（px）。** Easy の自作CNNは誤差2.8px → 完全一致 0.73（古典的手法と同じ）。ResNet18は 1.6px → **0.94**。
   - 検出率だけ見て「検出は完璧」と言うと、システムの性能を見誤ります。
3. **ResNet18（ImageNet学習済み）は、自作の小さなCNNより正確。** 四隅誤差は 2.8px → 1.6px、Medium の Det@0.9 は 0.39 → 0.81。**「モデルが大きいから」と「転移学習だから」の両方が効いています。** 学習済み重みなしの ResNet18（表の行。**1回だけの実験**）は、小さなCNNより良い（誤差2.3px）が、学習済みにはおよばない（1.6px）。同じモデルで比べると、**転移学習だけで、Medium の Det@0.9 が 0.57 → 0.81** に上がっています。
4. **Hard では、検出が成功（IoU 0.85）しても、読み取りはほぼ失敗。** 画像そのものの劣化（暗さ・ぼけ・ノイズ）で、**文字が読めない**ためです。ボトルネックは、**検出から文字認識（OCR）に移りました。** → 次の Phase3 で、OCRも学習するAIにします。

### 四隅の誤差が、どれくらい読み取りに響くか
全ての評価画像（ResNet18と小さなCNN、全シード）を、**四隅の誤差**で区切って、読み取りの成功率を見ます。

| 四隅の誤差（平均） | 枚数 | 読み取り完全一致 |
|---|---|---|
| 1px未満 | 164 | 0.994 |
| 1〜2px | 1361 | 0.865 |
| 2〜3px | 1364 | 0.587 |
| 3〜5px | 2067 | 0.189 |
| 5〜10px | 1948 | 0.008 |
| 10px以上 | 296 | 0.000 |

- **誤差が1px未満なら、ほぼ確実に読める（99%）。** 3〜5px で 19%、5px以上でほぼ 0%。
- つまり OCR は、**サブピクセル〜数ピクセルの位置精度**を要求します。Phase1 の「IoU 0.9以上が必要」と同じ事実を、より直接的に表しています。

### 実験: 正解の四隅に、わざと誤差を足してみる
上の関係は、**AIを使わなくても**確かめられます。**正解の四隅に、ランダムな誤差を足し**、Phase1 の OCR で読みます（誤差の大きさ σ ピクセル）。これが、**OCRが許す位置誤差の限界**を直接表します。

In [ ]:
rng = np.random.default_rng(0)
rows = []
for sigma in [0, 0.5, 1, 2, 3, 5]:                          # ← 演習で変えてみる
    ok = 0
    N = 60
    for i in range(N):
        s = generate_sample(split_seed("val", "easy", 100 + i), "easy")
        c = np.asarray(s.label["corners"]) + rng.normal(0, sigma, (4, 2))     # 四隅に誤差を足す
        ok += exact_match(read_text(rectify(s.image, c)).text, s.label["text"])
    rows.append({"四隅の誤差の大きさ σ(px)": sigma, "読み取り完全一致": ok / N})
tol = pd.DataFrame(rows).set_index("四隅の誤差の大きさ σ(px)")
tol.plot(marker="o", figsize=(5, 3), legend=False, ylim=(0, 1.05), title="四隅の誤差と、読み取りの成功率"); plt.grid(alpha=0.3)
tol

**見るポイント**: σが大きくなるにつれて、読み取りが**急に**失敗していきます。誤差1〜2pxあたりが分かれ目です。
これが「**検出器に求める精度**」の目安になります（Phase3 でOCRを学習するAIにすると、この許容範囲が広がる可能性があります）。

## 6. 学習データの作り方で、AIは変わる（分布シフト）

「**Easy（易しい条件）だけで学習**したAI」と「**全部の難易度で学習したAI**」を、同じ回数だけ学習して比べました（小さなCNN、3回）。

| 評価の難易度 | 学習データ | シード数 | 検出率(IoU≥0.5) | Det@0.9 | 平均IoU | 四隅誤差(px) |
|---|---|---|---|---|---|---|
| easy | Easy＋Medium＋Hard | 3 | 1.000 ±0.000 | 0.954 ±0.045 | 0.944 ±0.008 | 2.782 ±0.614 |
| easy | Easyだけ | 3 | 1.000 ±0.000 | 0.851 ±0.124 | 0.932 ±0.016 | 3.544 ±1.050 |
| medium | Easy＋Medium＋Hard | 3 | 1.000 ±0.000 | 0.388 ±0.117 | 0.882 ±0.014 | 4.907 ±0.669 |
| medium | Easyだけ | 3 | 0.549 ±0.053 | 0.053 ±0.008 | 0.516 ±0.046 | 28.584 ±4.162 |
| hard | Easy＋Medium＋Hard | 3 | 0.989 ±0.008 | 0.040 ±0.016 | 0.763 ±0.016 | 8.320 ±0.776 |
| hard | Easyだけ | 3 | 0.018 ±0.007 | 0.000 ±0.000 | 0.063 ±0.013 | 81.861 ±0.505 |

- **Easy だけで学習したAIは、Easy では全部入りとほぼ同じ**（平均IoU 0.93 vs 0.94）ですが、**Medium では急に悪化**（検出率 1.00 → 0.55）し、**Hard ではほぼ失敗**（0.99 → 0.02、四隅誤差82px）。
- **学習した条件と、実際に使う条件が違う**と、AIは壊れます。これが**分布シフト**です。
- 古典的手法は、人間がしきい値を決めました。AIは、**学習データがその代わり**です。つまり、**「どんな条件の画像を学習データに入れるか」が、設計の中心**になります（合成データなら、難易度を自由に混ぜられます）。

## 7. 注意: このAIは「必ず1つ出す」
板が**ない**画像を入れると、どうなるでしょう。背景だけの画像を用意して、学習したAIに入れてみます。

In [ ]:
rng = np.random.default_rng(5)
bgs = [make_background(rng, t) for t in ["wall", "gradient", "car", "car"]]
x = torch.stack([to_tensor(b) for b in bgs]).to(device)
model.eval()
with torch.no_grad():
    q = model(x).cpu().numpy()
fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))
for ax, b, p in zip(axes, bgs, q):
    ax.imshow(b); ax.axis("off")
    c = target_to_corners(p); ax.plot(*np.vstack([c, c[:1]]).T, "-", color="red", lw=2)
fig.suptitle("板が無い画像でも、AIは「板の位置」を答えてしまう（赤）"); fig.tight_layout();

**見るポイント**: 板が写っていないのに、**四隅を出して**います（画像の内容にあまり関係なく、「板がありそうな平均的な位置」を答えています）。
- このAIは「必ず1枚ある」と習ったので、**「無い」と答える方法を知りません**。実際の駐車場では、板が写らない、遮られる、といった場合があります。
- だから、システムには「**この答えにどれくらい自信があるか**」（確信度）と、「**自信がなければ人に回す**」仕組みが必要です。これが **Phase4** のテーマです。
- Phase1 の古典的手法は、板が見つからなければ「未検出（None）」を返せました。**AIは精度が高い代わりに、「分からない」と言えない**のです。

## 🧪 試してみよう

### 演習1: 学習の長さを変える
**やること**: 「4. 学習してみる」のセルの **`EPOCHS = 6`** を **`EPOCHS = 2`**、**`12`** に書き換えて、そのセル以降を実行し直します（12は約40秒）。
- **見るポイント**: val のIoUは、エポックが増えるほど上がりますか？ どこで頭打ちになりますか？
- **分かること**: 学習が足りないと、位置が不正確。**読み取りに必要な精度（四隅誤差1〜2px）には、長い学習が必要**です。

### 演習2: Easyだけで学習して、Hardで試す（分布シフトを体験）
**やること**: 同じセルの **`TRAIN_DIFFS = ("easy", "medium", "hard")`** を **`("easy",)`** に書き換え、そのセル以降を実行し直します。**「学習したAIの予測を見る」**の図（特に Medium と Hard の行）を見ます。
- **見るポイント**: Hard の予測（赤）は、正解（緑）からどれくらい外れますか？ 表の「Easyだけ」と同じ傾向ですか？
- **分かること**: **学習データに入っていない条件には、対応できない**。
- ⚠ 終わったら、`("easy", "medium", "hard")` に戻してください。

### 演習3: 位置の誤差に対する、OCRの許容範囲
**やること**: 「実験: 正解の四隅に、わざと誤差を足してみる」のセルの **`[0, 0.5, 1, 2, 3, 5]`** を **`[0, 1, 1.5, 2, 2.5, 3]`** に書き換えて、実行します。
- **見るポイント**: 成功率が何%を切るのは、σがいくつのときですか？
- **分かること**: 検出器の**目標精度**（何pxまで許されるか）。

### 考えてみよう（コード不要）
1. **検出率（IoU≥0.5）が100%なのに、読み取りが失敗する**のは、検出の評価として何が足りなかったからでしょう？ 検出の評価に、どんな指標を足すべきですか？
2. 実際の駐車場の写真に、このAIをそのまま使ったら何が起きますか？（合成画像だけで学習しています。**sim2realギャップ**）
3. 「Hardでは検出できているのに読めない」システムを、駐車場に置くとき、**人に回す仕組み**（要確認）はどの段階に入れるべきでしょう？

---

## ✅ まとめ

- **学習するAI（四隅の回帰）は、検出（見つける）で古典的手法を圧倒した。** Hardでも検出率ほぼ100%。しきい値の手調整が要らず、1つのモデルで全条件に対応する。
- **しかし、読み取りには位置の精度（四隅誤差が1〜2px以内）が必要。** 検出率（IoU≥0.5）だけでは、システムの性能は分からない。**Det@0.9や四隅誤差**を見る。
- **転移学習（ImageNet学習済みのResNet18）**で、位置の精度が大きく向上した（四隅誤差が約半分）。
- **学習データの条件が、AIの得意・不得意を決める**（分布シフト）。合成データは、条件を自由に設計できる。
- Hard では、検出は成功しても**画像の劣化で文字が読めない**。ボトルネックは **OCR** に移った。
- **AIは「分からない」と言えない**（板が無くても答える）。**確信度と「要確認」**が必要（Phase4）。

➡ 次は **Phase3: 文字認識（OCR）**。Phase1 のテンプレートマッチを、**例から学ぶAI（CRNN＋CTC）**に置き換えます。位置のずれや劣化に強くなるかが焦点です。